# 02 — Feature Engineering

Builds the model-ready dataset for daily demand forecasting.

- **Input:** `data/raw/train.csv` — daily sales per store/item (10 stores × 50 items, 2013–2017)
- **Output:** `data/processed/features.parquet` — sales enriched with calendar, lag and rolling-window features

Each (store, item) pair is treated as an independent time series, so every lag/rolling feature is computed **within** its own series (grouped by `store` and `item`) to avoid mixing values across series.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/train.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["store", "item", "date"])

In [3]:
df.info()
df.nunique()

<class 'pandas.core.frame.DataFrame'>
Index: 913000 entries, 0 to 912999
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   date    913000 non-null  datetime64[ns]
 1   store   913000 non-null  int64         
 2   item    913000 non-null  int64         
 3   sales   913000 non-null  int64         
dtypes: datetime64[ns](1), int64(3)
memory usage: 34.8 MB


date     1826
store      10
item       50
sales     213
dtype: int64

## 1. Calendar features

Deterministic features derived from the date itself — always available at prediction time, no leakage risk. They let the model capture weekly seasonality (day of week), yearly seasonality (month / day of year) and payday-style effects (start/end of month).

In [4]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["dayofweek"] = df["date"].dt.dayofweek
df["dayofyear"] = df["date"].dt.dayofyear
df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)

df.head()

,date,store,item,sales,year,month,day,dayofweek,dayofyear,weekofyear,is_weekend,is_month_start,is_month_end
0,2013-01-01,1,1,13,2013,1,1,1,1,1,0,1,0
1,2013-01-02,1,1,11,2013,1,2,2,2,1,0,0,0
2,2013-01-03,1,1,14,2013,1,3,3,3,1,0,0,0
3,2013-01-04,1,1,13,2013,1,4,4,4,1,0,0,0
4,2013-01-05,1,1,10,2013,1,5,5,5,1,1,0,0


## 2. Lag features

Past sales of the same (store, item) series, shifted so that each row only sees information from **before** its own date. Lags are chosen to match the seasonality of daily retail sales:

- `7, 14, 28` — recent history aligned with weekly cycles
- `91, 182` — quarterly and half-year context
- `365` — same day last year (yearly seasonality)

In [5]:
LAGS = [7, 14, 28, 91, 182, 365]

series = df.groupby(["store", "item"])["sales"]
for lag in LAGS:
    df[f"sales_lag_{lag}"] = series.shift(lag)

df.filter(like="sales_lag").isna().mean().round(3)

sales_lag_7      0.004
sales_lag_14     0.008
sales_lag_28     0.015
sales_lag_91     0.050
sales_lag_182    0.100
sales_lag_365    0.200
dtype: float64

## 3. Rolling-window features

Moving averages and standard deviations summarize the recent level and volatility of each series. The window is computed over sales **shifted by 1 day**, so the current day's target is never included in its own features (this setup targets 1-day-ahead forecasting; for longer horizons, shift by the horizon instead).

In [6]:
WINDOWS = [7, 14, 28]

df["sales_shift1"] = df.groupby(["store", "item"])["sales"].shift(1)
shifted = df.groupby(["store", "item"])["sales_shift1"]

for window in WINDOWS:
    df[f"sales_rmean_{window}"] = shifted.transform(lambda s, w=window: s.rolling(w).mean())
    df[f"sales_rstd_{window}"] = shifted.transform(lambda s, w=window: s.rolling(w).std())

df = df.drop(columns="sales_shift1")
df.filter(like="sales_r").describe().round(2)

,sales_rmean_7,sales_rstd_7,sales_rmean_14,sales_rstd_14,sales_rmean_28,sales_rstd_28
count,909500.00,909500.00,906000.00,906000.00,899000.00,899000.00
mean,52.30,9.78,52.36,9.85,52.48,10.04
std,27.01,4.67,26.91,4.23,26.81,4.08
min,5.57,0.38,6.43,1.41,6.75,1.85
25%,30.43,6.23,30.57,6.57,30.71,6.81
50%,47.71,8.94,47.86,9.15,48.07,9.42
75%,69.43,12.55,69.50,12.46,69.64,12.66
max,175.43,37.78,167.79,33.02,164.89,30.02


## 4. Final dataset

The first 365 days of each series have missing lag values (there is no "one year ago" yet). We drop those warm-up rows — with 5 years of data per series, plenty of history remains — and persist the result as Parquet.

In [7]:
rows_before = len(df)
df_final = df.dropna().reset_index(drop=True)

print(f"Rows before: {rows_before:,}")
print(f"Rows after dropping warm-up period: {len(df_final):,}")
print(f"Features: {df_final.shape[1]} columns")
print(f"Date range: {df_final['date'].min().date()} → {df_final['date'].max().date()}")

df_final.to_parquet("../data/processed/features.parquet", index=False)
print("\nSaved to data/processed/features.parquet")

Rows before: 913,000
Rows after dropping warm-up period: 730,500
Features: 25 columns
Date range: 2014-01-01 → 2017-12-31



Saved to data/processed/features.parquet


## Summary

Starting from raw daily sales (`date, store, item, sales`), we built **21 features**:

| Group | Features |
|---|---|
| Calendar (9) | `year, month, day, dayofweek, dayofyear, weekofyear, is_weekend, is_month_start, is_month_end` |
| Lags (6) | `sales_lag_{7, 14, 28, 91, 182, 365}` |
| Rolling windows (6) | `sales_rmean_{7, 14, 28}`, `sales_rstd_{7, 14, 28}` |

All history-based features are leakage-safe: they only use values strictly before each row's date, within the same (store, item) series.

**Next:** `03_modeling.ipynb` — baselines (naive seasonal) and a LightGBM model with time-based validation.

> **Engineering note:** the logic developed step by step in this notebook was later promoted to [`src/features/build_features.py`](../src/features/build_features.py), parameterized by forecast horizon and covered by unit tests — including an [anti-leakage test](../tests/test_features.py) that verifies no feature can see data closer than `horizon` days to its target. Notebook [`04_evaluation.ipynb`](04_evaluation.ipynb) consumes that module directly.